In [1]:
import os
import re
import json
import glob
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama
import sys
from datetime import datetime
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
import pandas as pd

In [3]:
def load_file(file_path):
    """外部ファイルを安全に読み込む関数"""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"ファイルが見つかりません: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

def extract_project_summary(text):
    """【前処理の強化】会議URL、会議ID、Slackシステム通知などのノイズを完全に除外する関数"""
    
    # 💡 1. チャンネル設定やシステム通知によるノイズの除外
    system_noise = ["set the channel topic:", "set the channel description:", "has joined the channel"]
    if any(noise in text for noise in system_noise):
        return None
        
    # 💡 2. 面談用のWeb会議URLや会議ID、パスコードが含まれる投稿の強制除外
    # TeamsやZoom等のURL、および「会議 ID」「パスコード」という文字列を検知
    url_pattern = r"https://[a-zA-Z0-9\-\.]+(?:\.microsoft\.com|\.zoom\.us|\.webex\.com)[^\s>]*"
    has_meeting_url = re.search(url_pattern, text) is not None
    has_meeting_id = "会議 ID" in text or "パスコード" in text or "お待ち合わせ" in text
    
    if has_meeting_url or has_meeting_id:
        return None

    # 3. パターン1: 記号（=== や ---）に囲まれている部分を優先して探す
    symbol_pattern = r'(?:={3,}|-{3,})'
    matches = [m.start() for m in re.finditer(symbol_pattern, text)]
    
    if len(matches) >= 2:
        start_idx = text.find('\n', matches[0]) + 1
        end_idx = matches[-1]
        summary = text[start_idx:end_idx].strip()
        if len(summary) > 40:
            return summary

    # 4. パターン2: 記号がない場合、案件キーワードから末尾までを切り出す
    keyword_pattern = r'(【案件名】|案件：|概要：|【案件概要】|工程：)'
    match = re.search(keyword_pattern, text)
    if match:
        return text[match.start():].strip()
        
    # 本文があまりにも短い場合は案件ではないとみなす（50文字未満）
    if len(text.strip()) < 50:
        return None
        
    return text.strip()

# システムプロンプトの読み込み
prompt_path = os.path.join("prompt", "system_prompt.txt")
prompt_template = load_file(prompt_path)
print("✅ 【第1セル修正完了】会議情報・システム設定除外ロジックの実装完了。")

✅ 【第1セル修正完了】会議情報・システム設定除外ロジックの実装完了。


In [8]:
# test_dataフォルダ内の全ての.txtファイルを取得
test_files = sorted(glob.glob(os.path.join("test_data", "*.txt")))

# 切り出し結果を辞書に格納
extracted_summaries = {}

for file_path in test_files:
    file_name = os.path.basename(file_path)
    raw_post_text = load_file(file_path)
    summary = extract_project_summary(raw_post_text)
    
    if summary:
        print(f"✅ {file_name}: 切り出し成功（{len(summary)} 文字）")
        extracted_summaries[file_name] = summary
    else:
        print(f"❌ {file_name}: 切り出し失敗")

# 例として、2つ目のデータの切り出し中身をプレビュー表示してみる
if "slack_post_2.txt" in extracted_summaries:
    print("\n--- slack_post_2.txt の切り出し結果プレビュー ---")
    print(extracted_summaries["slack_post_2.txt"][:200] + "...")


✅ slack_post_1.txt: 切り出し成功（551 文字）
✅ slack_post_2.txt: 切り出し成功（197 文字）
✅ slack_post_3.txt: 切り出し成功（482 文字）
✅ slack_post_4.txt: 切り出し成功（462 文字）

--- slack_post_2.txt の切り出し結果プレビュー ---
案件名：需要予測サービスの運用業務
工程：保守・運用

場所：浜松町（※リモート応相談）
期間：6月～

スキル：
必須）
　　・SQLを用いたデータ抽出、加工経験（2年以上）
　　・Pythonでの開発経験（2年以上）

人数：1名
外国籍：不可　
精算：140-180h　
面談：2回（web）　　　　　
年齢：45歳まで

備考：個人事業主、フリーランス不可　
　　・9:00-18:00...


In [9]:
# Ollamaのgemma2:9bモデルを初期化
llm = Ollama(model="gemma2:9b", temperature=0.0)
prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
chain = prompt | llm

# 前のセルで切り出したデータを元にAI解析を実行
for file_name, summary in extracted_summaries.items():
    print(f"\n==========================================")
    print(f"🤖 AI解析実行中: {file_name}")
    print(f"==========================================")
    
    ai_result_json = chain.invoke({"text": summary})
    
    # 文字列の端にある余計な空白を削り、省略せずに強制出力する
    print(str(ai_result_json).strip())




🤖 AI解析実行中: slack_post_1.txt


C:\Users\numap\AppData\Local\Temp\ipykernel_31588\3681160677.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma2:9b", temperature=0.0)


{
    "project_name": "リサーチデータ利活用基盤の開発・保守（Python/SQL）",
    "summary": "リサーチ会社のデータ利活用基盤およびシステム基盤の開発・保守・運用業務です。PC・スマホのログやTV視聴ログなどのビッグデータを扱い、クレンジングや集計、それらを提供するWebアプリケーションの開発まで幅広く携わっていただきます。",
    "required_skills": "Pythonによるスクリプト作成、データ解析実務\nSQL（業務要件に基づいたクエリ作成、既存クエリの読解）\nLinux環境での詳細設計～テスト経験",
    "preferred_skills": "大量データ／ビッグデータのハンドリング経験（数億レコード規模の処理・集計の経験）\nSnowflake（他DWH製品（BigQuery、Redshift など）経験）\nTableau（ダッシュボード作成・データ可視化の実務経験）\nRuby、C#\nConfluence、JIRAなどAtlassian製品を用いたドキュメント管理、チケット管理\nバックエンド開発経験（バッチ処理、データ処理系が得意領域）",
    "period": "６月～長期",
    "location": "ひばりが丘",
    "hours": "09:00〜17:30（休憩1h）",
    "interview_count": "Web1回",
    "remarks": "週3〜4日リモート可\n140h-190h",
    "tools_and_languages": ["Python", "SQL", "Linux", "Snowflake", "BigQuery", "Redshift", "Tableau", "Ruby", "C#", "Confluence", "JIRA"]
}

🤖 AI解析実行中: slack_post_2.txt
{
    "project_name": "需要予測サービスの運用業務",
    "summary": "保守・運用",
    "required_skills": "SQLを用いたデータ抽出、加工経験（2年以上）\nPythonでの開発経験（2年以上）",
    "preferred_skill

In [ ]:
# ==========================================
# 設定情報 (本来はGCPの環境変数やSecret Managerから取得)
# ==========================================
# 実際のトークンがある場合はここに設定します（例: "xoxb-12345..."）
SLACK_BOT_TOKEN = os.environ.get("SLACK_BOT_TOKEN", "YOUR_BOT_TOKEN")

# トークンが未設定（初期状態）ならテスト用のダミーモードで動かすフラグ
IS_MOCK_MODE = SLACK_BOT_TOKEN == "YOUR_BOT_TOKEN"

# Slackクライアントの初期化
slack_client = WebClient(token=SLACK_BOT_TOKEN)

def get_user_profile_by_id(user_id):
    """
    【機能要件3.3 & 4.0】
    SlackのユーザーID（U12345678など）から
    「表示名(本名)」と「メールアドレス」を取得する関数
    """
    if IS_MOCK_MODE:
        # --- テスト用のダミーデータを返す（モックモード） ---
        if user_id.startswith("U_SALES"): # 営業担当者の場合
            return {"display_name": "好井 太郎", "email": "yoshii@example.com"}
        else: # メンションされた技術社員の場合
            return {"display_name": "田中 次郎", "email": "tanaka@example.com"}
            
    try:
        # 実際のSlack APIを叩いてプロフィールを取得
        response = slack_client.users_info(user=user_id)
        user_info = response["user"]
        
        # 表示名（設定されていなければ本名）を取得
        profile = user_info.get("profile", {})
        display_name = profile.get("display_name") or user_info.get("real_name") or "名前なし"
        email = profile.get("email", "None")
        
        return {"display_name": display_name, "email": email}
        
    except SlackApiError as e:
        print(f"Slack APIエラー: {e.response['error']}")
        return {"display_name": "エラーにより取得失敗", "email": "None"}

# ==========================================
# テスト実行
# ==========================================
if __name__ == "__main__":
    print(f"--- Slack API 連携テスト (モード: {'モック動作' if IS_MOCK_MODE else '本番接続'}) ---")
    
    # 1. 投稿主（営業）のIDを仮定してメールアドレスと本名を取得
    mock_sales_id = "U_SALES_001"
    sales_profile = get_user_profile_by_id(mock_sales_id)
    print(f"👤 営業（投稿主）名 : {sales_profile['display_name']}")
    print(f"📧 営業メールアドレス: {sales_profile['email']}")
    print("-" * 30)
    
    # 2. 案件にメンションされた技術社員のIDを仮定して本名を取得
    mock_engineer_id = "U_ENG_999"
    engineer_profile = get_user_profile_by_id(mock_engineer_id)
    print(f"🎯 提案対象の社員名  : {engineer_profile['display_name']}")


--- Slack API 連携テスト (モード: モック動作) ---
👤 営業（投稿主）名 : 好井 太郎
📧 営業メールアドレス: yoshii@example.com
------------------------------
🎯 提案対象の社員名  : 田中 次郎


In [ ]:

# ==========================================
# 🔐 【本番設定】取得した本物の情報に書き換えてください
# ==========================================
SLACK_BOT_TOKEN = ""
CHANNEL_ID = "C06GFCRTQCW"

# Slackクライアントを本物のトークンで初期化
slack_client = WebClient(token=SLACK_BOT_TOKEN)

def get_real_slack_history(channel_id):
    """本物のSlackから直近100件の投稿履歴を直接取得する関数"""
    try:
        print(f"🛰️ Slackからチャンネル({channel_id})の過去ログを取得中...")
        response = slack_client.conversations_history(channel=channel_id, limit=300)
        return response["messages"]
    except SlackApiError as e:
        print(f"❌ Slack APIエラーが発生しました: {e.response['error']}")
        if e.response['error'] == "not_in_channel":
            print("💡 対策: Slack画面でに対象チャンネルにアプリを招待（/invite @アプリ名）してください。")
        elif e.response['error'] == "invalid_auth":
            print("💡 対策: SLACK_BOT_TOKEN（xoxb-...）の値が正しいか確認してください。")
        return []

def get_user_display_name(user_id):
    """Slack上のユーザーID（Uxxxx）から、プロフィールに登録されている『表示名（本名）』を直接取得する"""
    try:
        res = slack_client.users_info(user=user_id)
        user_info = res["user"]
        profile = user_info.get("profile", {})
        
        # 表示名(有野 亘人など)を取得。未設定ならリアルネームを使用
        display_name = profile.get("display_name") or user_info.get("real_name") or user_id
        return display_name
    except SlackApiError:
        return user_id

# ==========================================
# 🚀 実行メイン処理
# ==========================================
if __name__ == "__main__":
    # --- 💡 期間の条件設定 ---
    # 指定された日時を数値（UNIXタイムスタンプ）に変換
    start_date = datetime(2026, 5, 1, 0, 0, 0)
    end_date = datetime(2026, 6, 1, 0, 0, 0)
    
    start_ts = start_date.timestamp()
    end_ts = end_date.timestamp()
    
    print(f"📅 検索期間を設定しました: {start_date} から {end_date} まで\n")

    # 1. 本物のSlackからメッセージリストを取得
    messages = get_real_slack_history(CHANNEL_ID)
    
    print(f"📥 合計 {len(messages)} 件の投稿を読み込みました。精査を開始します。\n")
    match_count = 0
    
    for msg in messages:
        # Slackのタイムスタンプ（文字列型）を取得して、小数点より前（秒数）を数値（浮動小数点）に変換
        msg_ts_str = msg.get("ts", "0")
        msg_ts = float(msg_ts_str)
        
        # 💡 【期間判定】投稿時間が指定した5月1日〜6月1日の範囲外なら、処理をスキップ
        if not (start_ts <= msg_ts < end_ts):
            continue
            
        # 期間内の場合のみ、日本時間に変換して画面表示用の文字列を作る
        posted_time = datetime.fromtimestamp(msg_ts).strftime('%Y/%m/%d %H:%M:%S')
        
        text = msg.get("text", "")
        
        # メンション（<@Uxxxxxx>）を正規表現で探す
        mention_pattern = r"<@([A-Za-z0-9_]+)>"
        found_user_ids = re.findall(mention_pattern, text)
        
        if found_user_ids:
            for u_id in found_user_ids:
                real_name = get_user_display_name(u_id)
                
                # 「有野 亘人」さんへのメンションかどうかを判定
                if real_name == "有野 亘人":
                    match_count += 1
                    print(f"==================================================")
                    print(f"🎯 【有野 亘人さん宛て：2026年5月の投稿を発見！】")
                    print(f"==================================================")
                    print(f"⏰ 投稿日時（日本時間）: {posted_time}")
                    print(f"📄 本文（生のテキスト）:\n{text}\n")
                    
    print(f"🏁 精査が完了しました。期間内の対象案件は合計 {match_count} 件でした。")


📅 検索期間: 2026-05-01 00:00:00 から 2026-06-01 00:00:00 まで
🎯 探索対象: @有野 亘人

🛰️ Slackからチャンネル(C06GFCRTQCW)の過去ログを取得中...

🏁 精査完了。条件に合致する純粋な投稿が 【33件】 見つかりました。


In [7]:
# 検証のため凍結
'''
def save_raw_english_csv(data_row, post_time_float):
    """英語ヘッダーのまま生データを保存・上書きする関数"""
    dt = datetime.fromtimestamp(post_time_float)
    fiscal_year = dt.year if dt.month >= 4 else dt.year - 1
    
    file_year = f"raw_debug_analysis_{fiscal_year}.csv"
    file_month = f"raw_debug_analysis_{dt.strftime('%Y%m')}.csv"
    
    new_df = pd.DataFrame([data_row])
    
    for file_name in [file_year, file_month]:
        if os.path.exists(file_name):
            existing_df = pd.read_csv(file_name)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df.to_csv(file_name, index=False, encoding="utf-8-sig")
        else:
            new_df.to_csv(file_name, index=False, encoding="utf-8-sig")
        print(f"📁 生データをCSVに保存・上書きしました: {file_name}")

if __name__ == "__main__":
    print("=== 🚀 処理開始：厳選されたメッセージのAI解析 ＆ 英語ヘッダー生保存 ===")
    
    # 1. AIの初期化
    llm = Ollama(model="gemma2:9b", temperature=0.0)
    prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
    chain = prompt | llm
    
    keys_10 = ["project_name", "summary", "required_skills", "preferred_skills", "period", "location", "hours", "interview_count", "remarks", "tools_and_languages"]
    
    debug_logs = []
    saved_count = 0
    
    # 💡 前段のセルで厳選された「filtered_messages」をそのまま処理する
    if 'filtered_messages' not in locals() or not filtered_messages:
        print("ℹ️ 解析対象のメッセージ（filtered_messages）が0件、または存在しません。処理を終了します。")
    else:
        for msg in filtered_messages:
            text = msg.get("text", "")
            ts_float = float(msg.get("ts", "0"))
            dt = datetime.fromtimestamp(ts_float)
            
            # 前段でメッセージ内に埋め込んだ対象者名（有野 亘人など）を取得
            target_user_name = msg.get("resolved_target_user", "不明な社員")
            
            # 💡 会議URLや会議ID、システム設定通知などの面談ノイズを関数側で検知して弾く
            summary_text = extract_project_summary(text)
            
            log_entry = {
                "投稿時間": dt.strftime('%Y/%m/%d %H:%M:%S'),
                "投稿全文(冒頭30文字)": text.strip().replace("\n", " ")[:30] + "...",
                "判定結果": "採用（純粋な案件情報の可能性大）" if summary_text is not None else "除外（会議URL・設定ノイズ等）",
                "切り出されたテキスト(冒頭30文字)": summary_text.strip().replace("\n", " ")[:30] + "..." if summary_text else "None"
            }
            debug_logs.append(log_entry)
            
            # 会議URLやIDが含まれる面談連絡は、ここで確実にスキップ（完全除外）
            if summary_text is None:
                continue
                
            print(f"\n🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間={log_entry['投稿時間']}")
            ai_output = chain.invoke({"text": summary_text})
            
            try:
                ai_data = json.loads(str(ai_output).strip())
                
                row_data = {}
                row_data["raw_post_text"] = text
                
                # 営業確認用
                for k in keys_10:
                    val = ai_data.get(k, "None")
                    row_data[f"edited_{k}"] = ", ".join(val) if isinstance(val, list) else val
                
                row_data["price"] = ""
                row_data["leader_name"] = ""
                row_data["slack_url"] = f"https://slack.com{CHANNEL_ID}/p{msg.get('ts').replace('.', '')}"
                row_data["target_user"] = target_user_name  # 💡 動的に特定された名前をセット
                row_data["sales_user"] = get_user_display_name(msg.get("user"))
                
                # AI初期抽出データ
                for k in keys_10:
                    val = ai_data.get(k, "None")
                    row_data[f"ai_{k}"] = ", ".join(val) if isinstance(val, list) else val
                    
                row_data["slack_timestamp"] = msg.get("ts")
                
                # 保存を実行
                save_raw_english_csv(row_data, ts_float)
                saved_count += 1
                
            except Exception as e:
                continue

        # 検証用ログをCSV保存
        debug_df = pd.DataFrame(debug_logs)
        debug_df.to_csv("切り分け検証.csv", index=False, encoding="utf-8-sig")
        
        print("\n==========================================")
        print(f"🏁 処理が完了しました！")
        print(f"📦 厳選され保存された純粋な案件数: {saved_count} 件")
        print("==========================================")
'''        


'\ndef save_raw_english_csv(data_row, post_time_float):\n    """英語ヘッダーのまま生データを保存・上書きする関数"""\n    dt = datetime.fromtimestamp(post_time_float)\n    fiscal_year = dt.year if dt.month >= 4 else dt.year - 1\n\n    file_year = f"raw_debug_analysis_{fiscal_year}.csv"\n    file_month = f"raw_debug_analysis_{dt.strftime(\'%Y%m\')}.csv"\n\n    new_df = pd.DataFrame([data_row])\n\n    for file_name in [file_year, file_month]:\n        if os.path.exists(file_name):\n            existing_df = pd.read_csv(file_name)\n            combined_df = pd.concat([existing_df, new_df], ignore_index=True)\n            combined_df.to_csv(file_name, index=False, encoding="utf-8-sig")\n        else:\n            new_df.to_csv(file_name, index=False, encoding="utf-8-sig")\n        print(f"📁 生データをCSVに保存・上書きしました: {file_name}")\n\nif __name__ == "__main__":\n    print("=== 🚀 処理開始：厳選されたメッセージのAI解析 ＆ 英語ヘッダー生保存 ===")\n\n    # 1. AIの初期化\n    llm = Ollama(model="gemma2:9b", temperature=0.0)\n    prompt = PromptTemplate(tem

In [8]:
def save_raw_english_csv(data_row, post_time_float):
    """英語ヘッダーのまま、動的な成果物CSVファイルへ直接保存・上書きする関数"""
    dt = datetime.fromtimestamp(post_time_float)
    fiscal_year = dt.year if dt.month >= 4 else dt.year - 1
    
    # 成果物のファイル名を動的に指定
    file_year = f"データ分析案件_{fiscal_year}.csv"
    file_month = f"データ分析案件_{dt.strftime('%Y%m')}.csv"
    
    # 辞書データから直接DataFrameを作成
    new_df = pd.DataFrame([data_row])
    
    for file_name in [file_year, file_month]:
        if os.path.exists(file_name):
            # ファイルがあれば読み込んで末尾に追記（上書き保存）
            existing_df = pd.read_csv(file_name)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df.to_csv(file_name, index=False, encoding="utf-8-sig")
        else:
            # なければ新規作成
            new_df.to_csv(file_name, index=False, encoding="utf-8-sig")
        print(f"💾 CSVファイルにデータを上書き保存しました: {file_name}")

if __name__ == "__main__":
    print("=== 🚀 処理開始：正規版 案件解析＆成果物CSV直接保存パイプライン ===")
    
    # 1. AIの初期化
    llm = Ollama(model="gemma2:9b", temperature=0.0)
    prompt = PromptTemplate(template=prompt_template, input_variables=["text"])
    chain = prompt | llm
    
    keys_10 = ["project_name", "summary", "required_skills", "preferred_skills", "period", "location", "hours", "interview_count", "remarks", "tools_and_languages"]
    saved_count = 0
    
    # 前段のセルで厳選された「filtered_messages」をそのまま処理
    if 'filtered_messages' not in locals() or not filtered_messages:
        print("ℹ️ 解析対象のメッセージ（filtered_messages）が0件、または存在しません。処理を終了します。")
    else:
        for msg in filtered_messages:
            text = msg.get("text", "")
            ts_float = float(msg.get("ts", "0"))
            dt = datetime.fromtimestamp(ts_float)
            
            target_user_name = msg.get("resolved_target_user", "不明な社員")
            
            # 会議URLや会議ID、システム設定通知などの面談ノイズを関数側で検知して弾く
            summary_text = extract_project_summary(text)
            
            # 会議URLやIDが含まれる面談連絡は、ここで確実にスキップ（完全除外）
            if summary_text is None:
                continue
                
            print(f"\n🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間={dt.strftime('%Y/%m/%d %H:%M:%S')}")
            ai_output = chain.invoke({"text": summary_text})
            
            try:
                ai_data = json.loads(str(ai_output).strip())
                
                row_data = {}
                row_data["raw_post_text"] = text
                
                # 営業確認用
                for k in keys_10:
                    val = ai_data.get(k, "None")
                    row_data[f"edited_{k}"] = ", ".join(val) if isinstance(val, list) else val
                
                row_data["price"] = ""
                row_data["leader_name"] = ""
                row_data["slack_url"] = f"https://slack.com{CHANNEL_ID}/p{msg.get('ts').replace('.', '')}"
                row_data["target_user"] = target_user_name  
                row_data["sales_user"] = get_user_display_name(msg.get("user"))
                
                # AI初期抽出データ
                for k in keys_10:
                    val = ai_data.get(k, "None")
                    row_data[f"ai_{k}"] = ", ".join(val) if isinstance(val, list) else val
                    
                row_data["slack_timestamp"] = msg.get("ts")
                
                # 成果物CSVへダイレクトに保存を実行
                save_raw_english_csv(row_data, ts_float)
                saved_count += 1
                
            except Exception as e:
                continue

        print("\n==========================================")
        print(f"🏁 すべての処理が完了しました！")
        print(f"📦 生成された成果物内の純粋な案件数: {saved_count} 件")
        print("==========================================")

=== 🚀 処理開始：正規版 案件解析＆成果物CSV直接保存パイプライン ===

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/22 11:03:03
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/21 13:50:30
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/20 16:57:40
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/20 14:31:28
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/20 14:31:11
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/20 14:06:39
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv

🎯 純粋な案件投稿を検知。AI解析を実行します: 投稿時間=2026/05/19 15:36:33
💾 CSVファイルにデータを上書き保存しました: データ分析案件_2026.csv
💾 CSVファイルにデータを上書き保存しました: データ分析案件_202605.csv
